# Phase 2 — Waveform, Spectrogram & Audio Features

```
WAV -> Waveform -> STFT -> Spectrogram -> Mel filter bank -> Mel spectrogram -> Features
```

**A GPU is NOT needed for this phase.** These are small signal-processing calculations that run in about a second on a CPU.
You can leave the Colab runtime on "CPU".

## 1. Check PyTorch (GPU optional)

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())   # False is fine for Phase 2
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Clone the repository and install requirements

In [ ]:
import os
if not os.path.exists("/content/Voice_engine"):
    !git clone https://github.com/8919134556/Voice_engine.git /content/Voice_engine
%cd /content/Voice_engine
!git pull
!pip install -q -r requirements.txt

## 3. Access your private dataset

**Option A — Google Drive (recommended).** Expected location:
`MyDrive/voice_engine/dataset/speaker_01/audio_001.wav` (and 002, 003).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATASET_DIR = "/content/drive/MyDrive/voice_engine/dataset"
!ls "{DATASET_DIR}/speaker_01"

**Option B — upload for this session only** (skip if you used Drive).

In [ ]:
# Run ONLY if you are not using Google Drive.
import os, shutil
from google.colab import files
DATASET_DIR = "/content/Voice_engine/dataset"
os.makedirs(f"{DATASET_DIR}/speaker_01", exist_ok=True)
for name in files.upload():
    shutil.move(name, f"{DATASET_DIR}/speaker_01/{name}")
!ls "{DATASET_DIR}/speaker_01"

## 4. Run the whole Phase 2 analysis with one command

In [ ]:
!python scripts/analyze_audio.py --dataset "{DATASET_DIR}"

In [ ]:
# Show the saved plots for audio_001
from IPython.display import Image, display
for kind in ["waveform", "spectrogram", "mel"]:
    display(Image(f"outputs/phase2/audio_001_{kind}.png"))

---
# Step by step — what the script does inside

### Part 1 — Load (Phase 1 code) → mono, 16 kHz tensor

In [ ]:
from pathlib import Path
from src.audio.loader import load_mono_16k, list_audio_files
from src.audio import features as F
from src.visualization import plots

wav_files = list_audio_files(Path(DATASET_DIR) / "speaker_01")
path = wav_files[0]
waveform, sr = load_mono_16k(path)

print(path.name)
print("Sample rate:", sr)
print("Samples:    ", waveform.shape[1])
print(f"Duration:    {waveform.shape[1] / sr:.2f} s")
print("Shape:      ", waveform.shape)

### Part 2 — Waveform: amplitude over time
Flat line = silence. Big swings = loud speech. Dense small wiggles = hissy sounds like "s".

In [ ]:
plots.plot_waveform(waveform, sr, f"Waveform — {path.name}");

In [ ]:
# Zoom in on 30 ms in the middle: you can see the repeating pattern of a voiced sound
mid = waveform.shape[1] // 2
plots.plot_waveform(waveform[:, mid:mid + int(0.03 * sr)], sr, "30 ms zoom");

### Part 3 — STFT → spectrogram
Cut into 25 ms windows every 10 ms, measure frequencies in each window.

In [ ]:
power = F.stft_power(waveform)
print("Spectrogram shape:", power.shape, "= (frequency bins, time frames)")
print("Frequency bins: 0 Hz ...", sr // 2, "Hz in steps of", sr / F.N_FFT, "Hz")
print("Time frames:    one every", F.HOP_LENGTH / sr * 1000, "ms")
plots.plot_spectrogram(F.power_to_db(power), sr, title=f"Spectrogram — {path.name}");

### Part 4 — Mel filter bank → Mel spectrogram
Each triangle below collects energy from a range of frequencies.
Narrow triangles at low Hz (detail where our ears care), wide ones at high Hz.

In [ ]:
fbank = F.mel_filterbank(sr)
print("Filter bank shape:", fbank.shape, "= (frequency bins, Mel bands)")
plots.plot_mel_filterbank(fbank, sr);

In [ ]:
mel = F.mel_spectrogram(waveform, sr)
print("Mel spectrogram shape:", mel.shape, "= (Mel bands, time frames)")
plots.plot_mel_spectrogram(F.power_to_db(mel), sr, title=f"Mel spectrogram — {path.name}");

### Part 5 + 6 — Basic features and comparison table

In [ ]:
rows = []
for p in wav_files:
    w, s = load_mono_16k(p)
    rows.append({"filename": p.name, **F.extract_features(w, s)})

print(f"{'filename':<16}{'duration':>10}{'RMS':>10}{'ZCR':>10}{'centroid Hz':>13}")
for r in rows:
    print(f"{r['filename']:<16}{r['duration']:>10.2f}{r['rms']:>10.4f}{r['zcr']:>10.4f}{r['spectral_centroid']:>13.1f}")

### Part 7 — The CSV saved by the script
To keep it, copy it to Drive: `!cp outputs/phase2/phase2_audio_features.csv /content/drive/MyDrive/voice_engine/`

In [ ]:
print(open("outputs/phase2/phase2_audio_features.csv").read())

### Mini experiment — see the features react
Make the audio quieter and add hiss, and watch what happens to RMS, ZCR and the centroid.

In [ ]:
quiet = waveform * 0.3
hissy = waveform + 0.02 * torch.randn_like(waveform)
for name, w in [("original", waveform), ("30% volume", quiet), ("+ hiss", hissy)]:
    f = F.extract_features(w, sr)
    print(f"{name:<12} RMS={f['rms']:.4f}  ZCR={f['zcr']:.4f}  centroid={f['spectral_centroid']:.0f} Hz")